
# Priority Product Test Group Creation

## Objective

Create five test groups of OD markets for Priority product testing such that the groups are as comparable as possible.

Because the Priority product is assigned at the OD market level, the balancing objective should focus on market characteristics rather than flight-level or offer-level metrics.

The primary goal is to ensure that each treatment group contains a similar mix of market types, reducing the risk that observed differences in performance are driven by differences in market composition rather than the treatment itself.

---

## Experimental Unit

**Unit of assignment:** OD Market

Examples:

- DFW-MIA
- CLT-MCO
- ORD-DCA

Each OD market will be assigned to exactly one test group.

---

## Desired Balance Criteria

The five groups should be balanced with respect to the following characteristics:

### 1. Number of OD Markets

Each group should contain approximately the same number of OD markets.

Example:

| Group | OD Count |
|---------|---------:|
| Control | ~20% |
| T1 | ~20% |
| T2 | ~20% |
| T3 | ~20% |
| T4 | ~20% |

The goal is to avoid test groups that are substantially larger or smaller than others.

---

### 2. Priority Group Mix

Each treatment arm should contain a similar distribution of Priority Groups.

Example diagnostics:

| Group | PG1 | PG2 | PG3 | PG4 | PG5 | PG6 |
|---------|-----:|-----:|-----:|-----:|-----:|-----:|
| Control | x% | x% | x% | x% | x% | x% |
| T1 | x% | x% | x% | x% | x% | x% |
| T2 | x% | x% | x% | x% | x% | x% |
| T3 | x% | x% | x% | x% | x% | x% |
| T4 | x% | x% | x% | x% | x% | x% |

No treatment group should be disproportionately concentrated in a specific Priority Group.

---

### 3. Flight Duration Mix

Each treatment arm should contain a similar distribution of flight-duration categories.

Example diagnostics:

| Group | Ultra Short | Short | Medium | Long |
|---------|------------:|-------:|-------:|------:|
| Control | x% | x% | x% | x% |
| T1 | x% | x% | x% | x% |
| T2 | x% | x% | x% | x% |
| T3 | x% | x% | x% | x% |
| T4 | x% | x% | x% | x% |

The objective is to avoid systematic differences in trip length across treatment groups.

---

### 4. DOW Profile Mix

The goal is not necessarily to balance individual departures by weekday.

Instead, the goal is to balance the mix of market-level DOW profiles.

Examples:

- Weekday-heavy markets
- Weekend-heavy markets
- Balanced markets

Possible diagnostics:

| Group | Avg Mon | Avg Tue | Avg Wed | Avg Thu | Avg Fri | Avg Sat | Avg Sun |
|---------|---------:|---------:|---------:|---------:|---------:|---------:|---------:|
| Control | x% | x% | x% | x% | x% | x% | x% |
| T1 | x% | x% | x% | x% | x% | x% | x% |
| T2 | x% | x% | x% | x% | x% | x% | x% |
| T3 | x% | x% | x% | x% | x% | x% | x% |
| T4 | x% | x% | x% | x% | x% | x% | x% |

Alternatively, DOW profiles may be collapsed into broader weekday-versus-weekend measures if that provides a more interpretable balancing target.

---

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import itertools

RANDOM_SEED = 42

GROUPS = [
    "Control",
    "T1_minus15",
    "T2_minus10",
    "T3_plus10",
    "T4_plus15"
]

DOW_COLS = [
    "pct_sun",
    "pct_mon",
    "pct_tue",
    "pct_wed",
    "pct_thu",
    "pct_fri",
    "pct_sat"
]

N_FOLDS = 10

TREATMENT_VALUE_MAP = {
    "Control": 1.00,
    "T1_minus15": 0.85,
    "T2_minus10": 0.90,
    "T3_plus10": 1.10,
    "T4_plus15": 1.15
}

# July 14 2026: Need to be adjusted. Ask Sam about how long exploration runs in general
START_DATE = "2026-08-01"
END_DATE = "2026-08-14"

In [0]:
priority_airport = pd.read_excel("Priority_Group_Airport.xlsx")

priority_airport_spark = spark.createDataFrame(priority_airport)
priority_airport_spark.createOrReplaceTempView("priority_airport")

finalTransactionOfferSale = spark.table("rm_workspace.finalTransactionOfferSale_B")
finalTransactionOfferSale.createOrReplaceTempView("finalTransactionOfferSale")

print(f"finalTransactionOfferSale_B: {finalTransactionOfferSale.count():,} rows")

In [0]:
itinerary = (
    spark.table("rm_workspace.tmp_pnr_spine")
    .filter(F.length(F.col("fare_basis_cd")) <= 8)
    .filter(F.col("pax_count") > 0)
)

itinerary.createOrReplaceTempView("itinerary")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW offer_sale AS
SELECT
    f.*,
    dayofweek(f.OD_dep_dt) AS DOW,
    COALESCE(p.Group, 6) AS Priority_Group
FROM finalTransactionOfferSale f
LEFT JOIN priority_airport p
    ON f.od_origin = p.Airport;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW od_dow AS

WITH dow_counts AS (
    SELECT
        od_origin_airprt_iata_cd AS od_origin,
        od_destntn_airprt_iata_cd AS od_destination,
        dayofweek(od_local_dep_dt) AS dow,
        COUNT(*) AS dep_cnt
    FROM itinerary
    WHERE region LIKE '%US48%'
    GROUP BY
        od_origin_airprt_iata_cd,
        od_destntn_airprt_iata_cd,
        dayofweek(od_local_dep_dt)
),

totals AS (
    SELECT
        od_origin,
        od_destination,
        SUM(dep_cnt) AS total_dep
    FROM dow_counts
    GROUP BY
        od_origin,
        od_destination
)

SELECT
    d.od_origin,
    d.od_destination,

    ROUND(100.0 * SUM(CASE WHEN dow = 1 THEN dep_cnt ELSE 0 END) / MAX(total_dep), 2) AS pct_sun,
    ROUND(100.0 * SUM(CASE WHEN dow = 2 THEN dep_cnt ELSE 0 END) / MAX(total_dep), 2) AS pct_mon,
    ROUND(100.0 * SUM(CASE WHEN dow = 3 THEN dep_cnt ELSE 0 END) / MAX(total_dep), 2) AS pct_tue,
    ROUND(100.0 * SUM(CASE WHEN dow = 4 THEN dep_cnt ELSE 0 END) / MAX(total_dep), 2) AS pct_wed,
    ROUND(100.0 * SUM(CASE WHEN dow = 5 THEN dep_cnt ELSE 0 END) / MAX(total_dep), 2) AS pct_thu,
    ROUND(100.0 * SUM(CASE WHEN dow = 6 THEN dep_cnt ELSE 0 END) / MAX(total_dep), 2) AS pct_fri,
    ROUND(100.0 * SUM(CASE WHEN dow = 7 THEN dep_cnt ELSE 0 END) / MAX(total_dep), 2) AS pct_sat

FROM dow_counts d
JOIN totals t
    ON d.od_origin = t.od_origin
   AND d.od_destination = t.od_destination

GROUP BY
    d.od_origin,
    d.od_destination;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW od_volume AS

WITH pnr_od AS (
    SELECT
        od_origin_airprt_iata_cd AS od_origin,
        od_destntn_airprt_iata_cd AS od_destination,
        pnr_loctr_id,
        MAX(pax_count) AS pax_count
    FROM itinerary
    WHERE region LIKE '%US48%'
    GROUP BY
        od_origin_airprt_iata_cd,
        od_destntn_airprt_iata_cd,
        pnr_loctr_id
)

SELECT
    od_origin,
    od_destination,
    COUNT(DISTINCT pnr_loctr_id) AS pnr_cnt,
    SUM(pax_count) AS pax_cnt
FROM pnr_od
GROUP BY
    od_origin,
    od_destination;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW od_market_features AS

SELECT
    o.od_origin,
    o.od_destination,

    MAX(o.Priority_Group) AS priority_group,
    MAX(o.FlightDuration) AS flight_duration,

    d.pct_sun,
    d.pct_mon,
    d.pct_tue,
    d.pct_wed,
    d.pct_thu,
    d.pct_fri,
    d.pct_sat,

    COALESCE(v.pnr_cnt, 0) AS pnr_cnt,
    COALESCE(v.pax_cnt, 0) AS pax_cnt

FROM offer_sale o

LEFT JOIN od_dow d
    ON o.od_origin = d.od_origin
   AND o.od_destination = d.od_destination

LEFT JOIN od_volume v
    ON o.od_origin = v.od_origin
   AND o.od_destination = v.od_destination

WHERE o.region_group = 'Domestic'

GROUP BY
    o.od_origin,
    o.od_destination,
    d.pct_sun,
    d.pct_mon,
    d.pct_tue,
    d.pct_wed,
    d.pct_thu,
    d.pct_fri,
    d.pct_sat,
    v.pnr_cnt,
    v.pax_cnt;

In [0]:
od_features = spark.sql("""
SELECT
    od_origin,
    od_destination,
    priority_group,
    flight_duration,
    pct_sun,
    pct_mon,
    pct_tue,
    pct_wed,
    pct_thu,
    pct_fri,
    pct_sat,
    pnr_cnt,
    pax_cnt
FROM od_market_features
""").toPandas()

# Force numeric types after Spark -> pandas conversion
for c in DOW_COLS:
    od_features[c] = pd.to_numeric(od_features[c], errors="coerce")

for c in ["pnr_cnt", "pax_cnt"]:
    od_features[c] = pd.to_numeric(od_features[c], errors="coerce").fillna(0)

# Optional: if any OD has missing DOW because od_dow did not join
od_features[DOW_COLS] = od_features[DOW_COLS].fillna(0)

display(od_features[DOW_COLS].describe())


od_features["stratum"] = (
    od_features["priority_group"].astype(str)
    + "_"
    + od_features["flight_duration"].astype(str)
)

od_features = (
    od_features
    .groupby("stratum", group_keys=False)
    .apply(lambda x: x.sample(frac=1, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

od_features["group_idx"] = (
    od_features
    .groupby("stratum")
    .cumcount()
    % len(GROUPS)
)

od_features["test_group"] = (
    od_features["group_idx"]
    .map(dict(enumerate(GROUPS)))
)

In [0]:
def validate_assignment(df, title="BALANCE SUMMARY"):
    print("=" * 70)
    print(title)
    print("=" * 70)

    count_balance = df.groupby("test_group").size()
    print("\nOD count by group:")
    print(count_balance)
    print("\nOD count max-min:")
    print(count_balance.max() - count_balance.min())

    pg_balance = pd.crosstab(
        df["test_group"],
        df["priority_group"],
        normalize="index"
    ) * 100

    print("\nPriority Group mix:")
    display(pg_balance.round(2))

    print("\nPriority Group max-min percentage point imbalance:")
    print((pg_balance.max() - pg_balance.min()).round(3))

    fd_balance = pd.crosstab(
        df["test_group"],
        df["flight_duration"],
        normalize="index"
    ) * 100

    print("\nFlight Duration mix:")
    display(fd_balance.round(2))

    print("\nFlight Duration max-min percentage point imbalance:")
    print((fd_balance.max() - fd_balance.min()).round(3))

    dow_balance = df.groupby("test_group")[DOW_COLS].mean()

    print("\nDOW profile:")
    display(dow_balance.round(2))

    print("\nDOW max-min percentage point imbalance:")
    print((dow_balance.max() - dow_balance.min()).round(3))

    volume_summary = (
        df
        .groupby("test_group")
        .agg(
            n_ods=("od_origin", "size"),
            total_pnrs=("pnr_cnt", "sum"),
            avg_pnrs_per_od=("pnr_cnt", "mean"),
            median_pnrs_per_od=("pnr_cnt", "median"),
            total_pax=("pax_cnt", "sum"),
            avg_pax_per_od=("pax_cnt", "mean"),
            median_pax_per_od=("pax_cnt", "median")
        )
        .round(2)
    )

    print("\nVolume summary:")
    display(volume_summary)

    pnr_spread = volume_summary["total_pnrs"].max() - volume_summary["total_pnrs"].min()
    pnr_spread_pct = 100 * pnr_spread / volume_summary["total_pnrs"].mean()

    pax_spread = volume_summary["total_pax"].max() - volume_summary["total_pax"].min()
    pax_spread_pct = 100 * pax_spread / volume_summary["total_pax"].mean()

    print("\nPNR spread:")
    print(f"{pnr_spread:,.0f} PNRs ({pnr_spread_pct:.2f}% of mean)")

    print("\nPAX spread:")
    print(f"{pax_spread:,.0f} PAX ({pax_spread_pct:.2f}% of mean)")

    return {
        "count_balance": count_balance,
        "pg_balance": pg_balance,
        "fd_balance": fd_balance,
        "dow_balance": dow_balance,
        "volume_summary": volume_summary
    }


full_validation = validate_assignment(
    od_features,
    title="FULL UNIVERSE BALANCE SUMMARY"
)

In [0]:
final_assignment = od_features[
    [
        "od_origin",
        "od_destination",
        "priority_group",
        "flight_duration",
        "test_group",
        "pnr_cnt",
        "pax_cnt"
    ]
].copy()

final_assignment_spark = spark.createDataFrame(final_assignment)

final_assignment_spark.createOrReplaceTempView(
    "priority_test_group_assignment"
)


# final_assignment_spark.write.mode("overwrite").saveAsTable(
#     "rm_workspace.priority_test_group_assignment"
# )

In [0]:
od_features["pnr_bucket"] = pd.qcut(
    od_features["pnr_cnt"],
    q=10,
    labels=False,
    duplicates="drop"
)

fold_strata_cols = [
    "test_group",
    "priority_group",
    "flight_duration",
    "pnr_bucket"
]

df = od_features.copy()

rng = np.random.default_rng(RANDOM_SEED)
df["_rand"] = rng.random(len(df))

df = (
    df
    .sort_values(
        fold_strata_cols + ["pnr_cnt", "_rand"],
        ascending=[True, True, True, True, False, True]
    )
    .reset_index(drop=True)
)

df["_rank_in_stratum"] = (
    df
    .groupby(fold_strata_cols, dropna=False)
    .cumcount()
)

df["pilot_fold"] = df["_rank_in_stratum"] % N_FOLDS

In [0]:
fold_summary = (
    df
    .groupby(["test_group", "pilot_fold"])
    .agg(
        n_ods=("od_origin", "size"),
        total_pnrs=("pnr_cnt", "sum"),
        total_pax=("pax_cnt", "sum"),
        pct_sun=("pct_sun", "mean"),
        pct_mon=("pct_mon", "mean"),
        pct_tue=("pct_tue", "mean"),
        pct_wed=("pct_wed", "mean"),
        pct_thu=("pct_thu", "mean"),
        pct_fri=("pct_fri", "mean"),
        pct_sat=("pct_sat", "mean")
    )
    .reset_index()
)


best_combo = None
best_score = np.inf
best_combo_summary = None

for combo in itertools.product(range(N_FOLDS), repeat=len(GROUPS)):

    selected_rows = []

    for group, fold in zip(GROUPS, combo):
        row = fold_summary[
            (fold_summary["test_group"] == group)
            & (fold_summary["pilot_fold"] == fold)
        ]
        selected_rows.append(row)

    combo_summary = pd.concat(selected_rows, ignore_index=True)

    pnr_spread_pct = (
        100
        * (
            combo_summary["total_pnrs"].max()
            - combo_summary["total_pnrs"].min()
        )
        / combo_summary["total_pnrs"].mean()
    )

    pax_spread_pct = (
        100
        * (
            combo_summary["total_pax"].max()
            - combo_summary["total_pax"].min()
        )
        / combo_summary["total_pax"].mean()
    )

    count_spread_pct = (
        100
        * (
            combo_summary["n_ods"].max()
            - combo_summary["n_ods"].min()
        )
        / combo_summary["n_ods"].mean()
    )

    dow_spread_avg_pp = (
        combo_summary[DOW_COLS].max()
        - combo_summary[DOW_COLS].min()
    ).mean()

    score = (
        5.0 * pnr_spread_pct
        + 5.0 * pax_spread_pct
        + 1.0 * count_spread_pct
        + 2.0 * dow_spread_avg_pp
    )

    if score < best_score:
        best_score = score
        best_combo = combo
        best_combo_summary = combo_summary.copy()

print("=" * 70)
print("BEST PILOT FOLD COMBINATION")
print("=" * 70)
print("Best combo:")
print(dict(zip(GROUPS, best_combo)))
print(f"Best score: {best_score:.4f}")

display(best_combo_summary)

In [0]:
selected_parts = []

for group, fold in zip(GROUPS, best_combo):
    selected_parts.append(
        df[
            (df["test_group"] == group)
            & (df["pilot_fold"] == fold)
        ]
    )

pilot_od_features = (
    pd.concat(selected_parts, ignore_index=True)
    .reset_index(drop=True)
)

In [0]:
pilot_validation = validate_assignment(
    pilot_od_features,
    title="PILOT SUBSET BALANCE SUMMARY"
)

In [0]:
deployment_dates = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

n_dates = len(deployment_dates)

pilot_plan = pilot_od_features.copy()

pilot_plan = (
    pilot_plan
    .groupby("test_group", group_keys=False)
    .apply(lambda x: x.sample(frac=1, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

pilot_plan["date_idx"] = (
    pilot_plan
    .groupby("test_group")
    .cumcount()
    % n_dates
)

pilot_plan["Travel dates (Start)"] = pilot_plan["date_idx"].map(
    dict(enumerate(deployment_dates))
)

pilot_plan["Travel dates (End)"] = pilot_plan["Travel dates (Start)"]

pilot_plan["Loc1"] = (
    "P:" + pilot_plan["od_origin"].astype(str)
    + ",P:" + pilot_plan["od_destination"].astype(str)
)

pilot_plan["Comment"] = "Random Testing"
pilot_plan["Segment matches required"] = "FIRST"
pilot_plan["Result"] = pilot_plan["test_group"].map(TREATMENT_VALUE_MAP)

priority_plan = pilot_plan[
    [
        "Comment",
        "Segment matches required",
        "Travel dates (Start)",
        "Travel dates (End)",
        "Loc1",
        "Result"
    ]
].copy()

priority_plan["Travel dates (Start)"] = (
    pd.to_datetime(priority_plan["Travel dates (Start)"])
    .dt.strftime("%m/%d/%Y")
)

priority_plan["Travel dates (End)"] = (
    pd.to_datetime(priority_plan["Travel dates (End)"])
    .dt.strftime("%m/%d/%Y")
)

display(priority_plan)

In [0]:
priority_plan_spark = spark.createDataFrame(priority_plan)

priority_plan_spark.createOrReplaceTempView("priority_pilot_plan")

# priority_plan_spark.write.mode("overwrite").saveAsTable(
#     "rm_workspace.priority_pilot_plan"
# )

# priority_plan.to_csv(
#     "/dbfs/FileStore/priority_pilot_plan.csv",
#     index=False
# )